# Aula Completa: Streamlit + LLM com RAG + Deploy no Render

Este notebook foi reestruturado para servir como material de aula. A ordem da explicacao segue exatamente o fluxo pedido:

1. Sintaxe e estrutura do Streamlit.
2. Alteracoes feitas no projeto para inserir um LLM com RAG.
3. Passo a passo de deploy no Render.

O objetivo nao e apenas mostrar o codigo, mas explicar por que cada parte existe.

## 1) O que e Streamlit

O Streamlit e um framework Python para criar aplicacoes web interativas de forma declarativa.

A ideia principal e simples:

- o app e um script Python
- o script roda de cima para baixo
- toda interacao do usuario dispara uma nova execucao
- os widgets retornam valores Python
- `st.session_state` permite manter memoria entre as execucoes

Isso torna o Streamlit excelente para demos, MVPs, dashboards e aulas.

## 2) Sintaxe essencial do Streamlit

Abaixo esta um mapa da sintaxe mais importante, organizada por categoria.

### Configuracao da pagina

- `st.set_page_config(page_title=..., layout=...)`
- define titulo, layout, favicon e outros metadados

### Titulos e texto

- `st.title(...)`
- `st.header(...)`
- `st.subheader(...)`
- `st.caption(...)`
- `st.text(...)`
- `st.write(...)`
- `st.markdown(...)`

### Widgets de entrada

- `st.text_input(...)`
- `st.text_area(...)`
- `st.number_input(...)`
- `st.slider(...)`
- `st.selectbox(...)`
- `st.multiselect(...)`
- `st.radio(...)`
- `st.checkbox(...)`
- `st.toggle(...)`
- `st.button(...)`
- `st.file_uploader(...)`

### Layout

- `st.sidebar`
- `st.columns(...)`
- `st.tabs(...)`
- `st.container()`
- `st.expander(...)`

### Feedback visual

- `st.success(...)`
- `st.info(...)`
- `st.warning(...)`
- `st.error(...)`
- `st.spinner(...)`
- `st.progress(...)`

### Estado e cache

- `st.session_state`
- `@st.cache_data`
- `@st.cache_resource`

### Interface de chat

- `st.chat_input(...)`
- `st.chat_message(...)`

No projeto atual, os comandos mais importantes sao `st.sidebar`, `st.slider`, `st.selectbox`, `st.toggle`, `st.text_input`, `st.columns`, `st.write`, `st.caption` e `@st.cache_resource`.

In [ ]:
import streamlit as st

st.set_page_config(page_title="Exemplo Streamlit", layout="wide")

st.title("Demo Streamlit")
st.header("Cabecalho")
st.subheader("Subcabecalho")
st.caption("Essa legenda ajuda a contextualizar a tela.")
st.markdown("**Markdown** tambem funciona dentro do app.")

nome = st.text_input("Nome")
idade = st.slider("Idade", 0, 100, 25)
area = st.selectbox("Area", ["Dados", "IA", "Produto"])
ativo = st.toggle("Ativar modo demonstracao", value=True)

col1, col2 = st.columns(2)
with col1:
    st.write("Coluna 1")
with col2:
    st.write("Coluna 2")

with st.sidebar:
    st.header("Configuracoes")
    top_k = st.slider("Top-K", 1, 10, 3)

if ativo:
    st.success(f"Nome={nome or 'nao informado'} | idade={idade} | area={area} | top_k={top_k}")
else:
    st.info("Modo demonstracao desativado.")

## 3) Como pensar a reexecucao do Streamlit

Esse ponto costuma confundir alunos no inicio.

Quando o usuario mexe em um widget, o script roda de novo. Por isso, precisamos tomar alguns cuidados:

- recursos pesados devem ficar em cache
- conexoes e modelos podem usar `@st.cache_resource`
- dados transformados podem usar `@st.cache_data`
- estado de conversa, filtros ou historico podem ir para `st.session_state`

No projeto, o modelo de embeddings, o indice FAISS e o cliente do LLM foram tratados como recursos reutilizaveis.

In [ ]:
import streamlit as st

@st.cache_resource
def carregar_modelo_pesado():
    return "modelo carregado uma vez"

if "contador" not in st.session_state:
    st.session_state["contador"] = 0

if st.button("Incrementar"):
    st.session_state["contador"] += 1

st.write(carregar_modelo_pesado())
st.write("Contador:", st.session_state["contador"])

## 4) Arquitetura original do projeto

Antes da insercao do LLM, o projeto tinha uma arquitetura de busca semantica pura:

1. ler `docs.csv`
2. gerar embeddings com `SentenceTransformer`
3. criar um indice FAISS
4. receber a pergunta do usuario
5. buscar os Top-K documentos mais parecidos
6. mostrar esses documentos na tela

Esse desenho ja era didatico porque separava muito bem os conceitos de:

- base textual
- vetorizacao
- indexacao
- recuperacao

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import faiss

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

df = pd.read_csv("docs.csv")
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

texts = (df["title"] + "\n\n" + df["text"]).tolist()
emb = model.encode(texts, normalize_embeddings=True).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

pergunta = "Como solicitar reembolso?"
q = model.encode([pergunta], normalize_embeddings=True).astype("float32")
scores, ids = index.search(q, k=3)
scores, ids

## 5) O que e RAG na pratica

RAG significa `Retrieval-Augmented Generation`.

Na pratica, ele divide o sistema em duas partes:

- `retrieve`: buscar os documentos relevantes
- `generate`: gerar a resposta usando os documentos buscados

A diferenca central para o projeto original e que agora existe uma etapa de geracao controlada por contexto.

Fluxo final:

1. usuario faz uma pergunta
2. o sistema recupera os melhores trechos
3. o sistema monta um contexto textual
4. o LLM recebe pergunta + contexto
5. o LLM produz uma resposta consolidada
6. a resposta tenta citar as fontes recuperadas

Esse desenho reduz alucinacao porque o modelo e orientado a responder somente com base no FAQ.

## 6) Alteracoes feitas no `app.py` para inserir o LLM

As mudancas mais importantes foram estas:

### a) Separacao entre embeddings e LLM

Antes existia apenas o modelo de embeddings.
Agora temos dois componentes distintos:

- modelo de embeddings para recuperar documentos
- cliente do LLM para gerar a resposta final

### b) Novo carregamento do cliente do LLM

Foi criada a funcao `load_llm_client()`.
Ela verifica se existe `OPENAI_API_KEY` e, se existir, inicializa o cliente.

### c) Montagem de contexto

Foi criada a funcao `build_context(hits)`.
Ela transforma os Top-K resultados em um bloco textual com fonte, categoria, titulo e conteudo.

### d) Geracao da resposta final

Foi criada a funcao `generate_rag_answer(...)`.
Ela envia ao modelo:

- a pergunta do aluno
- o contexto recuperado
- instrucoes para responder em portugues
- instrucoes para nao inventar informacoes
- instrucoes para citar as fontes

### e) Fallback sem LLM

Se nao existir `OPENAI_API_KEY`, o app continua funcionando.
Nesse caso, o sistema usa um fallback que consolida os trechos recuperados em uma unica resposta.

### f) Controles na sidebar

Foram adicionados:

- toggle para ativar a geracao com LLM
- campo para informar o modelo
- mensagem indicando se a chave da API foi detectada

In [ ]:
def build_context(hits):
    blocks = []
    for i, hit in enumerate(hits, start=1):
        blocks.append(
            "\n".join([
                f"[Fonte {i}]",
                f"doc_id: {hit.get('doc_id')}",
                f"categoria: {hit.get('source')}",
                f"titulo: {hit.get('title')}",
                f"conteudo: {hit.get('text')}",
            ])
        )
    return "\n\n".join(blocks)

exemplo_hits = [
    {
        "doc_id": "1",
        "source": "financeiro",
        "title": "Reembolso: Qual o prazo para solicitar reembolso?",
        "text": "Voce pode solicitar reembolso em ate 7 dias corridos apos a primeira compra.",
    },
    {
        "doc_id": "29",
        "source": "financeiro",
        "title": "Reembolso: Como o estorno acontece?",
        "text": "No cartao, o estorno pode levar de 5 a 15 dias uteis. Para boleto, a transferencia e feita em conta indicada.",
    },
]

print(build_context(exemplo_hits))

## 7) Exemplo da chamada ao modelo

Abaixo esta a logica conceitual da chamada ao LLM.

O importante aqui e o papel do prompt:

- delimitar a tarefa
- restringir o conhecimento ao contexto recuperado
- orientar formato e idioma da resposta
- pedir citacoes de fonte

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

pergunta = "Qual o prazo para solicitar reembolso?"
contexto = build_context(exemplo_hits)

system_prompt = (
    "Voce responde perguntas de alunos com base exclusiva no contexto recuperado do FAQ. "
    "Se a resposta nao estiver no contexto, diga claramente que a base nao contem informacao suficiente. "
    "Seja objetivo, em portugues do Brasil, e cite as fontes como [Fonte 1], [Fonte 2]."
)

user_prompt = (
    f"Pergunta do aluno:\n{pergunta}\n\n"
    f"Contexto recuperado:\n{contexto}\n\n"
    "Gere uma unica resposta final consolidada, sem repetir blocos, seguida de uma linha com as fontes usadas."
)

# Exemplo de chamada. Descomente para testar com uma chave valida.
# response = client.chat.completions.create(
#     model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
#     temperature=0.2,
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": user_prompt},
#     ],
# )
# print(response.choices[0].message.content)

## 8) O que mudou visualmente na interface

A tela do app passou a ter dois blocos principais:

- `Contexto recuperado`: mostra os documentos encontrados pelo FAISS
- `Resposta final`: mostra a resposta consolidada

Didaticamente isso e muito bom porque permite explicar aos alunos a diferenca entre:

- o que foi recuperado
- o que foi gerado

Ou seja, a aula pode mostrar claramente onde termina a busca e onde comeca a geracao.

## 9) Estrutura recomendada 

1. abrir o `docs.csv` e explicar a base
2. abrir o `build_index.py` e explicar embeddings + FAISS
3. abrir o `app.py` e mostrar o `retrieve()`
4. mostrar a nova funcao `build_context()`
5. mostrar a funcao `generate_rag_answer()`
6. rodar o app sem chave para ver o fallback
7. rodar o app com chave para ver o LLM em acao

Essa progressao funciona bem porque vai do mais simples para o mais sofisticado.

## 10) Deploy no Render: conceito

O Render e uma plataforma de deploy gerenciado. Para esse projeto, ele vai executar o Streamlit como um servico web.

O que o Render precisa saber:

- como instalar as dependencias
- como iniciar a aplicacao
- quais variaveis de ambiente devem ser configuradas

No projeto, isso foi preparado com o arquivo `render.yaml`.

In [ ]:
render_yaml_exemplo = """
services:
  - type: web
    name: faq-empregadados
    runtime: python
    plan: free
    buildCommand: pip install -r requirements.txt
    startCommand: streamlit run app.py --server.address 0.0.0.0 --server.port $PORT
    envVars:
      - key: PYTHON_VERSION
        value: 3.11.11
      - key: OPENAI_MODEL
        value: gpt-4o-mini
"""

print(render_yaml_exemplo)

## 11) Deploy no Render: passo a passo

### Passo 1

Subir o projeto para GitHub ou outro provedor Git suportado.

### Passo 2

No Render, criar um novo `Web Service`.

### Passo 3

Selecionar o repositorio do projeto.

### Passo 4

Conferir se o Render detectou corretamente:

- `Build Command`: `pip install -r requirements.txt`
- `Start Command`: `streamlit run app.py --server.address 0.0.0.0 --server.port $PORT`

### Passo 5

Adicionar as variaveis de ambiente:

- `OPENAI_API_KEY`
- `OPENAI_MODEL` se quiser sobrescrever o padrao

### Passo 6

Fazer o deploy e abrir a URL publica.

### Passo 7

Testar se:

- o app carrega
- a busca semantica funciona
- a resposta final funciona quando a chave da API esta configurada

## 12) Cuidados importantes no deploy

Alguns pontos merecem explicacao em aula:

- `0.0.0.0` permite que o servico aceite conexoes externas
- `$PORT` e definido pelo proprio Render
- a chave `OPENAI_API_KEY` nunca deve ficar hardcoded no codigo
- o arquivo `data/faiss.index` precisa existir no deploy se o projeto depender dele
- se o indice nao existir, o usuario precisa rodar `python build_index.py` antes de subir ou no processo de build

No estado atual do projeto, como os arquivos em `data/` ja existem, o caminho mais simples e versiona-los junto com o repositorio.

## 13) Resumo final 

O projeto completo pode ser explicado assim:

- Streamlit cria a interface.
- O usuario digita uma pergunta.
- O modelo de embeddings transforma essa pergunta em vetor.
- O FAISS recupera os textos mais relevantes.
- O sistema monta um contexto com esses textos.
- O LLM gera uma unica resposta consolidada com base no contexto.
- O Render publica tudo isso como aplicacao web.

Essa sequencia mostra muito bem a diferenca entre:

- busca semantica
- RAG
- aplicacao interativa
- deploy em nuvem